In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>') 

/kaggle/input/competitions/mlp-kaggle-assignment-2-may-2026-term/sample_submission.csv
/kaggle/input/competitions/mlp-kaggle-assignment-2-may-2026-term/train.csv
/kaggle/input/competitions/mlp-kaggle-assignment-2-may-2026-term/test.csv


In [2]:
dfs = pd.read_csv("/kaggle/input/competitions/mlp-kaggle-assignment-2-may-2026-term/sample_submission.csv")
train = pd.read_csv("/kaggle/input/competitions/mlp-kaggle-assignment-2-may-2026-term/train.csv")
test = pd.read_csv("/kaggle/input/competitions/mlp-kaggle-assignment-2-may-2026-term/test.csv")

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns


sns.set_style('whitegrid')

# 1. Data Types

In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90000 entries, 0 to 89999
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                90000 non-null  int64  
 1   customer_id       90000 non-null  int64  
 2   last_name         90000 non-null  object 
 3   credit_score      80444 non-null  float64
 4   country           83979 non-null  object 
 5   gender            90000 non-null  object 
 6   age               90000 non-null  float64
 7   tenure            90000 non-null  int64  
 8   acc_balance       82743 non-null  float64
 9   prod_count        85137 non-null  float64
 10  has_card          90000 non-null  float64
 11  is_active         90000 non-null  float64
 12  estimated_salary  90000 non-null  float64
 13  exit_status       90000 non-null  int64  
dtypes: float64(7), int64(4), object(3)
memory usage: 9.6+ MB


# 2. Descriptive Statistic 

In [5]:
num_cols = ['credit_score','age','tenure','acc_balance','prod_count','has_card','is_active','estimated_salary']
train[num_cols].describe().T[['min','25%','50%','mean','75%','max','std']]

,min,25%,50%,mean,75%,max,std
credit_score,350.00,597.00,659.00,656.497054,710.0000,850.00,80.016856
age,18.00,32.00,37.00,38.119533,42.0000,92.00,8.855203
tenure,0.00,3.00,5.00,5.017022,7.0000,10.00,2.804813
acc_balance,0.00,0.00,0.00,55456.732147,119825.7500,250898.09,62788.474236
prod_count,1.00,1.00,2.00,1.552932,2.0000,4.00,0.548011
has_card,0.00,1.00,1.00,0.754289,1.0000,1.00,0.430510
is_active,0.00,0.00,0.00,0.497178,1.0000,1.00,0.499995
estimated_salary,11.58,74430.36,117505.07,112394.659679,154874.7875,199992.48,50360.440702


# 3. Missing Values - Identification & Handling

In [6]:
missing = train.isnull().sum()
missing_estimate = train.isnull().mean()*100
pd.DataFrame({'missing_count': missing, 'missing_estimate': missing_estimate})[missing>0]

,missing_count,missing_estimate
credit_score,9556,10.617778
country,6021,6.690000
acc_balance,7257,8.063333
prod_count,4863,5.403333


# Strategy:
    * credit_score, acc_balance, prod_count (numerical): impute with the median, which is robust to the skew we saw above (especially for acc_balance).
    * country (categorical): impute with the mode (most frequent category).




In [7]:
credit_score_median = train['credit_score'].median()
acc_balance_median = train['acc_balance'].median()
prod_count_median = train['prod_count'].median()
country_mode = train['country'].mode()[0]

for df in [train, test]:
    df['credit_score'] = df['credit_score'].fillna(credit_score_median)
    df['acc_balance']  = df['acc_balance'].fillna(acc_balance_median)
    df['prod_count']   = df['prod_count'].fillna(prod_count_median)
    df['country']      = df['country'].fillna(country_mode)

print('Remaining missing values in train:', train.isnull().sum().sum())
print('Remaining missing values in test :', test.isnull().sum().sum())

Remaining missing values in train: 0
Remaining missing values in test : 0


In [8]:
test.isnull().sum()

id                  0
customer_id         0
last_name           0
credit_score        0
country             0
gender              0
age                 0
tenure              0
acc_balance         0
prod_count          0
has_card            0
is_active           0
estimated_salary    0
dtype: int64

# 4. Duplicates — Identification & Handling

In [9]:
print('Fully duplicated rows:', train.duplicated().sum())

#check the id part as per observation it shows the duplicate or repeted values
print('Rows with a repeated customer_id:', train.duplicated(subset=['customer_id'], keep=False).sum())
print('Unique customer_ids:', train['customer_id'].nunique(), 'out of', len(train), 'rows')


#check duplicates ignoring only the row id column (in case the same coustomer record was logged twice
#under different row id) -- run AFTER imputation, since two rows that differed only in an
# originally-missing field can become identical once that field is imputed with the same value

cols_no_id = [c for c in train.columns if c != 'id']
n_dupe_rows = train.duplicated(subset= cols_no_id , keep=False).sum()
print("Rows involved in a full duplicate (ignoring id)")
train[train.duplicated(subset=cols_no_id, keep=False)].sort_values(cols_no_id).head(8)

Fully duplicated rows: 0
Rows with a repeated customer_id: 82608
Unique customer_ids: 18247 out of 90000 rows
Rows involved in a full duplicate (ignoring id)


,id,customer_id,last_name,credit_score,country,gender,age,tenure,acc_balance,prod_count,has_card,is_active,estimated_salary,exit_status
1700,1700,15591248,Chukwumaobim,628.0,France,Female,29.0,9,71996.29,2.0,1.0,0.0,34857.46,0
11438,11438,15591248,Chukwumaobim,628.0,France,Female,29.0,9,71996.29,2.0,1.0,0.0,34857.46,0
32655,32655,15643487,Sal,630.0,France,Male,39.0,10,105473.74,1.0,0.0,0.0,58854.88,1
44677,44677,15643487,Sal,630.0,France,Male,39.0,10,105473.74,1.0,0.0,0.0,58854.88,1
39619,39619,15672798,O'Brien,656.0,France,Female,45.0,7,0.00,2.0,1.0,1.0,199392.14,0
41434,41434,15672798,O'Brien,656.0,France,Female,45.0,7,0.00,2.0,1.0,1.0,199392.14,0
40992,40992,15705657,Hewitt,535.0,France,Female,44.0,2,0.00,2.0,1.0,1.0,136330.26,0
68920,68920,15705657,Hewitt,535.0,France,Female,44.0,2,0.00,2.0,1.0,1.0,136330.26,0


 There are no rows that are fully duplicated including id (each id is unique by construction). customer_id also repeats frequently (only 18,247 unique IDs across 90,000 rows) — checking those rows shows they mostly have different last_name, credit_score, age, balances

In [10]:
before = len(train)
train = train.drop_duplicates(subset=cols_no_id, keep='first').reset_index(drop=True)
print(f'Dropped {before - len(train)} duplicate rows. New train shape: {train.shape}')

Dropped 4 duplicate rows. New train shape: (89996, 14)
